# 00_Brain_project — 팀 공용 nnU-Net Experiment Manager FINAL

- 필수 확인 https://app.notion.com/p/nnU-NET-2-5D-Base-3a5ca6ebd6838071bdc9f8d86f90a99a

## 고정 기준

- 프로젝트 루트: `/content/drive/MyDrive/00_Brain_project`
- 개인 `nnU-Net 2.5D Exp05` → 팀 공용 `Base_Model`
- `Base_Model`은 영구 읽기 전용이며 실험번호를 사용하지 않는다.
- 신규 팀 실험은 `Exp01`, `Exp02`, `Exp03` 순서로 생성한다.
- 기존 이관 파일·Registry·경로·변수명을 재설계하거나 재생성하지 않는다.
- 모델 선택 기준은 **Validation Micro Dice**이다.
- Test는 `BASE_EVALUATE` 또는 최종 후보 `TEST_FINAL`에서만 사용한다.

## 실행 모드

| RUN_MODE | 기능 | Drive 저장 |
|---|---|---|
| `BASE_EVALUATE` | Base checkpoint Validation/Test 재현 | 없음 |
| `SMOKE` | ExpXX 2 epoch 연결 시험 | 있음 |
| `MANUAL` | 수동 파라미터 학습·Validation 평가 | 있음 |
| `RESUME` | 지정한 기존 RUN 이어서 학습·평가 | 있음 |
| `OPTUNA` | Optuna + 선택 Pruner | 있음 |
| `POSTPROCESS` | 기존 Validation probability 후처리 재탐색 | 있음 |
| `TEST_FINAL` | 지정 후보 최종 Test 1회 평가 | 있음 |

> 팀원은 **CELL 2 설정 셀만 수정**하는 것을 원칙으로 한다.


In [ ]:
# ==================================================================================================
# CELL 1. Google Drive 연결 및 고정 환경 확인
# ==================================================================================================

import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

NUMPY_VERSION = "2.0.2"
NNUNET_VERSION = "2.8.1"
OPTUNA_VERSION = "4.4.0"

ENV_MARKER = Path("/content/.brain_project_nnunet_environment_final")
ENV_SIGNATURE = (
    f"numpy={NUMPY_VERSION}|"
    f"nnunetv2={NNUNET_VERSION}|"
    f"optuna={OPTUNA_VERSION}"
)

required_versions = {
    "numpy": NUMPY_VERSION,
    "nnunetv2": NNUNET_VERSION,
    "optuna": OPTUNA_VERSION,
}

need_install = False
for package_name, expected_version in required_versions.items():
    try:
        installed_version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        need_install = True
        break
    if installed_version != expected_version:
        need_install = True
        break

if need_install:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        f"numpy=={NUMPY_VERSION}",
        f"nnunetv2=={NNUNET_VERSION}",
        f"optuna=={OPTUNA_VERSION}",
        "pandas",
        "nibabel",
        "scipy",
        "tqdm",
        "surface-distance",
    ])
    ENV_MARKER.write_text(ENV_SIGNATURE, encoding="utf-8")
    print("고정 버전 설치 완료. 런타임을 재시작한 후 CELL 1부터 다시 실행하세요.")
    os.kill(os.getpid(), 9)

import numpy as np
import torch

assert torch.cuda.is_available(), "Colab 런타임 유형을 GPU로 변경하세요."
assert shutil.which("nnUNetv2_train") is not None, "nnUNetv2_train 실행 파일이 없습니다."

print("=" * 100)
print("환경 확인")
print("=" * 100)
print("NumPy :", np.__version__)
print("nnU-Net:", importlib.metadata.version("nnunetv2"))
print("Optuna:", importlib.metadata.version("optuna"))
print("GPU    :", torch.cuda.get_device_name(0))
print("✅ CELL 1 완료")


In [ ]:
# --------------------------------------------------------------------------------------------------
# B. 실행 모드
# --------------------------------------------------------------------------------------------------
#
# 1) BASE_EVALUATE
#    - 이관된 Base_Model이 정상적으로 로딩되고 평가되는지 확인하는 모드
#    - Base checkpoint로 Validation과 Test 추론·평가를 다시 수행
#    - 기준 성능:
#        Validation Micro Dice ≈ 0.690981
#        Test Micro Dice       ≈ 0.712285
#    - /content의 임시 폴더만 사용
#    - Google Drive에 평가 결과, 로그, checkpoint, Exp 폴더를 저장하지 않음
#    - Base_Model과 Current Best를 수정하지 않음
#    - 팀 프로젝트를 처음 실행할 때 가장 먼저 사용하는 모드
#
# 2) SMOKE
#    - 전체 학습을 시작하기 전에 코드 연결 상태를 짧게 확인하는 시험 학습 모드
#    - Dataset, Custom Trainer, Base checkpoint, GPU 학습, Validation 평가,
#      저장 경로가 정상적으로 연결되는지 확인
#    - 현재 코드에서는 2 epoch, 적은 train/validation iteration으로 자동 축소
#    - ExpXX/manual_runs/RUN_XXX에 실제 결과를 저장함
#    - 학습 설정, checkpoint, console log, Validation 결과가 저장됨
#    - 기본값에서는 Experiment Best와 Current Best를 갱신하지 않음
#    - 단순 읽기 전용 TEST가 아니라 실제 짧은 학습이므로 Drive 저장이 발생함
#
# 3) MANUAL
#    - 사용자가 CELL 2에서 지정한 파라미터로 정식 학습하는 모드
#    - Loss, Optimizer, Learning Rate, Scheduler, Batch Size,
#      Oversampling, Epoch, Early Stopping 등을 직접 지정하여 실행
#    - USE_BASE_PRETRAINED_WEIGHTS=True:
#         Base_Model checkpoint를 초기 가중치로 사용하는 Fine-tuning
#    - USE_BASE_PRETRAINED_WEIGHTS=False:
#         Base 가중치를 사용하지 않는 Scratch 학습
#    - 새로운 RUN_XXX 번호를 자동 생성
#    - 학습 설정, checkpoint, console log, Validation 결과,
#      Threshold 및 Component Size 탐색 결과를 저장
#    - Validation Micro Dice가 기존 Best보다 높을 경우
#      Experiment Best와 Current Best를 갱신할 수 있음
#
# 4) RESUME
#    - 중단되었거나 추가 학습이 필요한 기존 Manual RUN을 이어서 학습하는 모드
#    - EXP_NO와 RESUME_RUN_ID로 이어서 실행할 RUN을 지정
#    - 예:
#         EXP_NO = 1
#         RESUME_RUN_ID = "RUN_000"
#    - 새로운 RUN 폴더를 만들지 않고 기존 RUN의 nnUNet_results를 사용
#    - nnUNetv2_train의 --c 옵션으로 checkpoint_latest.pth부터 재개
#    - 기존 RUN의 로그와 결과 경로에 이어서 기록됨
#    - 이미 완료된 실험을 새 조건으로 변경해 재사용하는 용도가 아니라,
#      동일 설정의 중단된 학습을 재개하는 용도
#
# 5) OPTUNA
#    - 여러 파라미터 조합을 자동 탐색하는 하이퍼파라미터 튜닝 모드
#    - SEARCH_*가 True인 변수만 Optuna 탐색 대상이 됨
#    - SEARCH_*가 False인 변수는 CELL 2의 수동값으로 고정됨
#    - Trial마다 독립된 TRIAL_XXX 폴더에 설정, checkpoint,
#      console log, Validation 평가 결과를 저장
#    - PRUNER_NAME에 따라 성능이 낮은 Trial을 조기에 중단할 수 있음
#    - Optuna Study는 SQLite DB에 저장되므로 같은 Study를 이어서 실행 가능
#    - 각 Trial의 최종 비교 기준은 Validation Micro Dice
#    - Test 데이터는 Optuna 탐색과 Best Trial 선택에 사용하지 않음
#
# 6) POSTPROCESS
#    - 기존 학습을 다시 하지 않고 저장된 Validation probability를 이용해
#      후처리 조건만 다시 탐색하는 모드
#    - THRESHOLD_CANDIDATES의 probability threshold와
#      MIN_COMPONENT_SIZE_CANDIDATES의 연결요소 제거 기준을 조합하여 평가
#    - Validation Micro Dice가 가장 높은 후처리 조합을 선택
#    - 모델 가중치와 checkpoint는 수정하지 않음
#    - 선택 대상은 TARGET_EXP_NO, TARGET_RUN_ID,
#      TARGET_SOURCE_KIND로 지정
#    - 새로운 후처리 탐색 결과는 대상 RUN 아래에 별도로 저장됨
#
# 7) TEST_FINAL
#    - Validation에서 최종 후보로 선택된 모델을 Test 데이터로
#      단 한 번 최종 평가하는 모드
#    - Validation에서 선택된 threshold와 minimum component size를
#      변경하지 않고 Test에 그대로 적용
#    - Test 결과로 Optuna 튜닝, threshold 재선택,
#      Current Best 선정을 다시 수행하면 안 됨
#    - Test case별, 환자별, 전체 요약 지표를 저장
#    - 대상 모델은 TARGET_EXP_NO, TARGET_RUN_ID,
#      TARGET_SOURCE_KIND로 지정
#    - 최종 일반화 성능 확인 및 보고서·발표용 평가에 사용
#
# 지원값:
# "BASE_EVALUATE" / "SMOKE" / "MANUAL" / "RESUME" /
# "OPTUNA" / "POSTPROCESS" / "TEST_FINAL"

In [ ]:
# ==================================================================================================
# CELL 2. 팀원이 수정하는 단일 설정 셀
# ==================================================================================================

# --------------------------------------------------------------------------------------------------
# A. 팀 신규 실험 번호
# Base_Model은 실험번호를 사용하지 않는다.
# --------------------------------------------------------------------------------------------------
EXP_NO = 1

# --------------------------------------------------------------------------------------------------
# B. 실행 모드
# BASE_EVALUATE / SMOKE / MANUAL / RESUME / OPTUNA / POSTPROCESS / TEST_FINAL
# --------------------------------------------------------------------------------------------------
RUN_MODE = "BASE_EVALUATE"

# --------------------------------------------------------------------------------------------------
# C. 실행 규모
# FAST / NORMAL / FULL / CUSTOM
# --------------------------------------------------------------------------------------------------
PROFILE = "FAST"

# --------------------------------------------------------------------------------------------------
# D. 입력 Context Slice
# 현재 Registry에 등록된 Context만 실행 가능하다.
# Base pretrained weight는 3-slice에서만 허용한다.
# --------------------------------------------------------------------------------------------------
NUM_CONTEXT_SLICES = 3

# --------------------------------------------------------------------------------------------------
# E. 초기화 방식
# True  : Base_Model checkpoint에서 fine-tuning
# False : Scratch 학습
# RESUME 모드에서는 기존 RUN checkpoint를 이어서 사용한다.
# --------------------------------------------------------------------------------------------------
USE_BASE_PRETRAINED_WEIGHTS = True
RESUME_RUN_ID = "RUN_000"

# --------------------------------------------------------------------------------------------------
# F. 학습 파라미터
# --------------------------------------------------------------------------------------------------
LOSS = "DiceCE"                 # DiceCE / Dice / Tversky / FocalTversky
OPTIMIZER = "AdamW"             # AdamW / Adam / SGD
LR = 5e-4
WEIGHT_DECAY = 1e-5
SCHEDULER = "PolyLR"            # PolyLR / Cosine / StepLR
BATCH_SIZE = 8
OVERSAMPLE_FOREGROUND_PERCENT = 0.33
POSITIVE_CASE_RATIO = 0.50

TVERSKY_ALPHA = 0.7             # FP 가중치
TVERSKY_BETA = 0.3              # FN 가중치
FOCAL_GAMMA = 2.0

FOLD = 0
RANDOM_SEED = 42

# --------------------------------------------------------------------------------------------------
# G. Epoch 및 Early Stopping
# PROFILE=CUSTOM일 때 CUSTOM_* 값이 사용된다.
# --------------------------------------------------------------------------------------------------
CUSTOM_N_TRIALS = 4
CUSTOM_EPOCHS = 100
CUSTOM_TOTAL_MINUTES = 480
CUSTOM_TRIAL_MINUTES = 180
CUSTOM_TRAIN_ITERS_PER_EPOCH = 150
CUSTOM_VAL_ITERS_PER_EPOCH = 25

EARLY_STOPPING_MIN_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 20
EARLY_STOPPING_MIN_DELTA = 0.001

# --------------------------------------------------------------------------------------------------
# H. Optuna 탐색 여부
# False 항목은 위 수동값으로 고정된다.
# --------------------------------------------------------------------------------------------------
SEARCH_LOSS = True
SEARCH_OPTIMIZER = True
SEARCH_LR = True
SEARCH_WEIGHT_DECAY = True
SEARCH_SCHEDULER = True
SEARCH_BATCH_SIZE = False
SEARCH_OVERSAMPLE = True
SEARCH_POSITIVE_CASE_RATIO = True
SEARCH_TVERSKY_ALPHA = True
SEARCH_FOCAL_GAMMA = True

SEARCH_SPACE = {
    "loss": ["DiceCE", "Tversky", "FocalTversky"],
    "optimizer": ["AdamW", "SGD"],
    "lr_low": 1e-4,
    "lr_high": 1e-2,
    "weight_decay_low": 1e-6,
    "weight_decay_high": 1e-3,
    "scheduler": ["PolyLR", "Cosine", "StepLR"],
    "batch_size": [4, 8, 12],
    "oversample_foreground_percent": [0.0, 0.33, 0.5],
    "positive_case_ratio": [0.4, 0.5, 0.6],
    "tversky_alpha": [0.6, 0.7, 0.8],
    "focal_gamma": [1.5, 2.0, 2.5],
}

OPTUNA_STAGE = 1

# NONE / MEDIAN / PERCENTILE / HYPERBAND / SUCCESSIVE_HALVING
PRUNER_NAME = "MEDIAN"
PRUNER_STARTUP_TRIALS = 2
PRUNER_WARMUP_STEPS = 10
PRUNER_INTERVAL_STEPS = 2
PRUNER_PERCENTILE = 50.0
PRUNER_MIN_RESOURCE = 5
PRUNER_REDUCTION_FACTOR = 3

# --------------------------------------------------------------------------------------------------
# I. Validation 후처리 탐색
# --------------------------------------------------------------------------------------------------
THRESHOLD_CANDIDATES = [0.30, 0.40, 0.50, 0.60, 0.70]
MIN_COMPONENT_SIZE_CANDIDATES = [0, 5, 10, 20, 50]
SLIDING_WINDOW_STEP_SIZE = 0.5

# --------------------------------------------------------------------------------------------------
# J. 저장·Best 관리
# --------------------------------------------------------------------------------------------------
BLOCK_DUPLICATE_MANUAL_CONFIG = False
BACKUP_PREVIOUS_BEST = True
UPDATE_EXPERIMENT_BEST = True
UPDATE_GLOBAL_BEST = True

# SMOKE는 구조 확인용이므로 기본적으로 Best 갱신에서 제외한다.
ALLOW_SMOKE_BEST_UPDATE = False

# --------------------------------------------------------------------------------------------------
# K. POSTPROCESS / TEST_FINAL 대상
# --------------------------------------------------------------------------------------------------
TARGET_EXP_NO = 1
TARGET_RUN_ID = "RUN_000"       # RUN_000 또는 TRIAL_000
TARGET_SOURCE_KIND = "MANUAL"   # MANUAL / OPTUNA / EXPERIMENT_BEST / CURRENT_BEST

# --------------------------------------------------------------------------------------------------
# L. 공통 nnU-Net 고정값
# --------------------------------------------------------------------------------------------------
NNUNET_CONFIGURATION = "2d"
PLANS_NAME = "nnUNetPlans"
CHECK_INTERVAL_SECONDS = 10

PROFILE_CONFIGS = {
    "FAST": {
        "n_trials": 2,
        "epochs": 10,
        "total_minutes": 60,
        "trial_minutes": 30,
        "train_iters": 35,
        "val_iters": 12,
    },
    "NORMAL": {
        "n_trials": 5,
        "epochs": 45,
        "total_minutes": 240,
        "trial_minutes": 90,
        "train_iters": 100,
        "val_iters": 20,
    },
    "FULL": {
        "n_trials": 20,
        "epochs": 100,
        "total_minutes": 720,
        "trial_minutes": 240,
        "train_iters": 250,
        "val_iters": 40,
    },
}

print("✅ CELL 2 설정 입력 완료")


In [ ]:
# ==================================================================================================
# CELL 3. 설정 검증 및 현재 확정 경로 연결
# ==================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import os
import random
import re
import shutil
import signal
import site
import subprocess
import sys
import tempfile
import time
from datetime import datetime
from pathlib import Path
from typing import Any

import nibabel as nib
import numpy as np
import pandas as pd
import torch
from scipy.ndimage import binary_erosion, distance_transform_edt
from scipy.ndimage import label as connected_label

RUN_MODE = RUN_MODE.upper()
PROFILE = PROFILE.upper()
PRUNER_NAME = PRUNER_NAME.upper()
TARGET_SOURCE_KIND = TARGET_SOURCE_KIND.upper()

VALID_RUN_MODES = {
    "BASE_EVALUATE",
    "SMOKE",
    "MANUAL",
    "RESUME",
    "OPTUNA",
    "POSTPROCESS",
    "TEST_FINAL",
}
assert RUN_MODE in VALID_RUN_MODES, f"지원하지 않는 RUN_MODE: {RUN_MODE}"
assert isinstance(EXP_NO, int) and EXP_NO >= 1
assert PROFILE in {"FAST", "NORMAL", "FULL", "CUSTOM"}
assert isinstance(NUM_CONTEXT_SLICES, int) and NUM_CONTEXT_SLICES >= 1 and NUM_CONTEXT_SLICES % 2 == 1
assert OPTUNA_STAGE >= 1
assert LOSS in {"DiceCE", "Dice", "Tversky", "FocalTversky"}
assert OPTIMIZER in {"AdamW", "Adam", "SGD"}
assert SCHEDULER in {"PolyLR", "Cosine", "StepLR"}
assert PRUNER_NAME in {"NONE", "MEDIAN", "PERCENTILE", "HYPERBAND", "SUCCESSIVE_HALVING"}
assert 0.0 <= OVERSAMPLE_FOREGROUND_PERCENT <= 1.0
assert 0.0 <= POSITIVE_CASE_RATIO <= 1.0
assert TVERSKY_ALPHA >= 0 and TVERSKY_BETA >= 0
assert abs((TVERSKY_ALPHA + TVERSKY_BETA) - 1.0) < 1e-6
assert BATCH_SIZE >= 1
assert EARLY_STOPPING_MIN_EPOCHS >= 0
assert EARLY_STOPPING_PATIENCE >= 1

if USE_BASE_PRETRAINED_WEIGHTS and RUN_MODE in {"SMOKE", "MANUAL", "OPTUNA"}:
    assert NUM_CONTEXT_SLICES == 3, (
        "Base_Model은 3-slice 2.5D 모델입니다. "
        "다른 Context에서는 USE_BASE_PRETRAINED_WEIGHTS=False를 사용하세요."
    )

if PROFILE == "CUSTOM":
    N_TRIALS = int(CUSTOM_N_TRIALS)
    TRAIN_EPOCHS = int(CUSTOM_EPOCHS)
    MAX_TOTAL_TRAINING_MINUTES = int(CUSTOM_TOTAL_MINUTES)
    MAX_TRIAL_MINUTES = int(CUSTOM_TRIAL_MINUTES)
    TRAIN_ITERS_PER_EPOCH = int(CUSTOM_TRAIN_ITERS_PER_EPOCH)
    VAL_ITERS_PER_EPOCH = int(CUSTOM_VAL_ITERS_PER_EPOCH)
else:
    profile_cfg = PROFILE_CONFIGS[PROFILE]
    N_TRIALS = int(profile_cfg["n_trials"])
    TRAIN_EPOCHS = int(profile_cfg["epochs"])
    MAX_TOTAL_TRAINING_MINUTES = int(profile_cfg["total_minutes"])
    MAX_TRIAL_MINUTES = int(profile_cfg["trial_minutes"])
    TRAIN_ITERS_PER_EPOCH = int(profile_cfg["train_iters"])
    VAL_ITERS_PER_EPOCH = int(profile_cfg["val_iters"])

if RUN_MODE == "SMOKE":
    N_TRIALS = 1
    TRAIN_EPOCHS = 2
    MAX_TOTAL_TRAINING_MINUTES = 30
    MAX_TRIAL_MINUTES = 20
    TRAIN_ITERS_PER_EPOCH = 5
    VAL_ITERS_PER_EPOCH = 2

MODEL_MODE = "2D" if NUM_CONTEXT_SLICES == 1 else "2.5D"

PROJECT_ROOT = Path("/content/drive/MyDrive/00_Brain_project")
PATHS = {
    "admin": PROJECT_ROOT / "00_admin",
    "data": PROJECT_ROOT / "01_data",
    "notebooks": PROJECT_ROOT / "02_notebooks",
    "docs": PROJECT_ROOT / "03_docs",
    "environment": PROJECT_ROOT / "04_environment",
    "nnunet": PROJECT_ROOT / "05_nnunet",
    "registry": PROJECT_ROOT / "06_model_registry",
    "experiments": PROJECT_ROOT / "07_experiments",
    "logs": PROJECT_ROOT / "08_logs",
    "deployment": PROJECT_ROOT / "09_deployment",
    "archive": PROJECT_ROOT / "10_archive",
}

for name, path in PATHS.items():
    assert path.exists(), f"기존 필수 프로젝트 폴더 없음 [{name}]: {path}"

NNUNET_ROOT = PATHS["nnunet"]
NNUNET_RAW = NNUNET_ROOT / "nnUNet_raw"
NNUNET_PREPROCESSED = NNUNET_ROOT / "nnUNet_preprocessed"
NNUNET_RESULTS = NNUNET_ROOT / "nnUNet_results"
CUSTOM_TRAINER_RUNTIME = NNUNET_ROOT / "custom_trainers_runtime"
OPTUNA_STUDIES_DIR = NNUNET_ROOT / "optuna_studies"

CONTEXT_REGISTRY_PATH = NNUNET_ROOT / "context_dataset_registry.json"
BASE_MODEL_DIR = PATHS["registry"] / "CT" / "base" / "Base_Model"
BASE_CHECKPOINT = BASE_MODEL_DIR / "model" / "checkpoint_best.pth"
BASE_TRAINER_FILE = BASE_MODEL_DIR / "trainer" / "nnUNetTrainerBHSD_Exp05_25DFinal.py"
BASE_METADATA_PATH = BASE_MODEL_DIR / "metadata" / "base_model_metadata.json"
BASE_METRICS_PATH = BASE_MODEL_DIR / "metadata" / "base_model_metrics.json"
BASE_MANIFEST_PATH = BASE_MODEL_DIR / "metadata" / "manifest.json"
BASE_PROTECTION_PATH = BASE_MODEL_DIR / "DO_NOT_OVERWRITE.txt"
BEST_MODELS_DIR = PATHS["registry"] / "CT" / "best"
CURRENT_BEST_DIR = BEST_MODELS_DIR / "current"
CURRENT_BEST_JSON = CURRENT_BEST_DIR / "current_best.json"
BEST_HISTORY_DIR = BEST_MODELS_DIR / "history"

for required in [
    CONTEXT_REGISTRY_PATH,
    BASE_CHECKPOINT,
    BASE_TRAINER_FILE,
    BASE_METADATA_PATH,
    BASE_METRICS_PATH,
    BASE_MANIFEST_PATH,
    BASE_PROTECTION_PATH,
    CURRENT_BEST_JSON,
]:
    assert required.is_file(), f"기존 필수 파일 없음: {required}"

registry_payload = json.loads(CONTEXT_REGISTRY_PATH.read_text(encoding="utf-8"))
dataset_entry = registry_payload.get("datasets", {}).get(str(NUM_CONTEXT_SLICES))
assert dataset_entry is not None, (
    f"{NUM_CONTEXT_SLICES}-slice Dataset이 Registry에 없습니다. "
    f"현재 등록: {sorted(registry_payload.get('datasets', {}).keys())}"
)

if isinstance(dataset_entry, list):
    available = [item for item in dataset_entry if item.get("available", True)]
    assert available, f"{NUM_CONTEXT_SLICES}-slice 사용 가능 Dataset이 없습니다."
    if NUM_CONTEXT_SLICES == 3:
        base_items = [item for item in available if item.get("dataset_name") == "Dataset001_BHSD_25D"]
        dataset_item = base_items[0] if base_items else available[0]
    else:
        dataset_item = available[0]
else:
    dataset_item = dataset_entry

DATASET_ID = int(dataset_item["dataset_id"])
DATASET_NAME = str(dataset_item["dataset_name"])
SELECTED_RAW_DATASET = NNUNET_RAW / DATASET_NAME
SELECTED_PREPROCESSED_DATASET = NNUNET_PREPROCESSED / DATASET_NAME

for required in [
    SELECTED_RAW_DATASET / "dataset.json",
    SELECTED_RAW_DATASET / "imagesTr",
    SELECTED_RAW_DATASET / "labelsTr",
    SELECTED_PREPROCESSED_DATASET / "dataset.json",
    SELECTED_PREPROCESSED_DATASET / "dataset_fingerprint.json",
    SELECTED_PREPROCESSED_DATASET / "nnUNetPlans.json",
    SELECTED_PREPROCESSED_DATASET / "splits_final.json",
    SELECTED_PREPROCESSED_DATASET / f"{PLANS_NAME}_{NNUNET_CONFIGURATION}",
]:
    assert required.exists(), f"기존 학습 필수 항목 없음: {required}"

dataset_json = json.loads((SELECTED_RAW_DATASET / "dataset.json").read_text(encoding="utf-8"))
channel_names = dataset_json.get("channel_names", dataset_json.get("modality", {}))
INPUT_CHANNEL_INDICES = sorted(int(key) for key in channel_names.keys())

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
os.environ["nnUNet_compile"] = "false"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

BASE_RESULT_MODEL_DIR = (
    NNUNET_RESULTS
    / "Dataset001_BHSD_25D"
    / "nnUNetTrainerBHSD_Exp05_25DFinal__nnUNetPlans__2d"
)

assert BASE_RESULT_MODEL_DIR.is_dir(), (
    "Base 평가용 nnUNet_results 모델 폴더가 없습니다: "
    f"{BASE_RESULT_MODEL_DIR}"
)

print("=" * 110)
print("현재 확정 프로젝트 연결")
print("=" * 110)
print("PROJECT_ROOT        :", PROJECT_ROOT)
print("RUN_MODE            :", RUN_MODE)
print("EXP_NO              :", EXP_NO)
print("DATASET             :", DATASET_NAME)
print("CONTEXT             :", NUM_CONTEXT_SLICES)
print("BASE CHECKPOINT     :", BASE_CHECKPOINT)
print("BASE RESULT MODEL   :", BASE_RESULT_MODEL_DIR)
print("TRAIN EPOCHS        :", TRAIN_EPOCHS)
print("TRAIN/VAL ITERS     :", TRAIN_ITERS_PER_EPOCH, "/", VAL_ITERS_PER_EPOCH)
print("✅ CELL 3 완료")


In [ ]:
# ==================================================================================================
# CELL 4. 공통 함수: JSON, SHA256, 경로·RUN 선택, 평가 지표
# ==================================================================================================

def write_json(path: Path, payload: Any) -> None:
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def config_hash(payload: dict[str, Any]) -> str:
    encoded = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:12]

def next_run_number(manual_runs_dir: Path) -> int:
    values = []
    for path in manual_runs_dir.glob("RUN_*"):
        try:
            values.append(int(path.name.split("_")[1]))
        except Exception:
            pass
    return max(values, default=-1) + 1

def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator > 0 else np.nan

def confusion_counts(gt, pred):
    gt = np.asarray(gt).astype(bool)
    pred = np.asarray(pred).astype(bool)
    tp = int(np.logical_and(gt, pred).sum())
    fp = int(np.logical_and(~gt, pred).sum())
    fn = int(np.logical_and(gt, ~pred).sum())
    tn = int(np.logical_and(~gt, ~pred).sum())
    return tp, fp, fn, tn

def metrics_from_counts(tp, fp, fn, tn):
    return {
        "Dice": safe_divide(2 * tp, 2 * tp + fp + fn),
        "IoU": safe_divide(tp, tp + fp + fn),
        "Precision": safe_divide(tp, tp + fp),
        "Recall": safe_divide(tp, tp + fn),
        "Specificity": safe_divide(tn, tn + fp),
    }

def patient_id_from_case_name(file_name):
    return re.sub(r"_z\d+\.nii\.gz$", "", file_name)

def hd95_2d(gt, pred, spacing_xy):
    gt = np.squeeze(gt).astype(bool)
    pred = np.squeeze(pred).astype(bool)
    if gt.ndim != 2 or pred.ndim != 2 or not gt.any() or not pred.any():
        return np.nan
    gt_surface = np.logical_xor(gt, binary_erosion(gt))
    pred_surface = np.logical_xor(pred, binary_erosion(pred))
    distance_to_gt = distance_transform_edt(~gt_surface, sampling=spacing_xy)
    distance_to_pred = distance_transform_edt(~pred_surface, sampling=spacing_xy)
    distances = np.concatenate([
        distance_to_gt[pred_surface],
        distance_to_pred[gt_surface],
    ])
    return float(np.percentile(distances, 95)) if distances.size else np.nan

def evaluate_prediction_folder(prediction_dir, ground_truth_dir, split_name):
    rows = []
    prediction_dir = Path(prediction_dir)
    ground_truth_dir = Path(ground_truth_dir)

    for prediction_path in sorted(prediction_dir.glob("*.nii.gz")):
        gt_path = ground_truth_dir / prediction_path.name
        if not gt_path.is_file():
            continue
        prediction_nii = nib.load(str(prediction_path))
        gt_nii = nib.load(str(gt_path))
        pred = np.asarray(prediction_nii.dataobj) > 0
        gt = np.asarray(gt_nii.dataobj) > 0
        tp, fp, fn, tn = confusion_counts(gt, pred)
        metric = metrics_from_counts(tp, fp, fn, tn)
        gt_positive = bool(gt.any())
        pred_positive = bool(pred.any())
        slice_dice = (
            np.nan if not gt_positive and not pred_positive
            else safe_divide(2 * tp, 2 * tp + fp + fn)
        )
        spacing_xy = tuple(float(v) for v in gt_nii.header.get_zooms()[:2])
        rows.append({
            "Split": split_name,
            "Case": prediction_path.name,
            "Patient": patient_id_from_case_name(prediction_path.name),
            "GT_positive": gt_positive,
            "Prediction_positive": pred_positive,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "Slice_Dice_nonempty_union": slice_dice,
            "Slice_IoU": metric["IoU"],
            "Slice_Precision": metric["Precision"],
            "Slice_Recall": metric["Recall"],
            "HD95_mm_both_positive": hd95_2d(gt, pred, spacing_xy),
        })

    case_df = pd.DataFrame(rows)
    assert not case_df.empty, f"{split_name} 평가 대상 prediction이 없습니다: {prediction_dir}"

    total_tp = int(case_df["TP"].sum())
    total_fp = int(case_df["FP"].sum())
    total_fn = int(case_df["FN"].sum())
    total_tn = int(case_df["TN"].sum())
    micro = metrics_from_counts(total_tp, total_fp, total_fn, total_tn)

    lesion_slice_df = case_df[case_df["GT_positive"]]
    union_positive_df = case_df[case_df["GT_positive"] | case_df["Prediction_positive"]]
    normal_slice_df = case_df[~case_df["GT_positive"]]
    patient_counts = case_df.groupby("Patient", as_index=False)[["TP", "FP", "FN", "TN"]].sum()

    patient_rows = []
    for _, row in patient_counts.iterrows():
        metric = metrics_from_counts(
            int(row["TP"]),
            int(row["FP"]),
            int(row["FN"]),
            int(row["TN"]),
        )
        patient_rows.append({"Patient": row["Patient"], **metric})
    patient_df = pd.DataFrame(patient_rows)

    summary_df = pd.DataFrame([{
        "Split": split_name,
        "Number_of_slices": len(case_df),
        "Number_of_patients": case_df["Patient"].nunique(),
        "GT_lesion_slices": int(case_df["GT_positive"].sum()),
        "GT_empty_slices": int((~case_df["GT_positive"]).sum()),
        "Micro_Dice": micro["Dice"],
        "Micro_IoU": micro["IoU"],
        "Micro_Precision": micro["Precision"],
        "Micro_Recall": micro["Recall"],
        "Specificity": micro["Specificity"],
        "Lesion_slice_Macro_Dice": lesion_slice_df["Slice_Dice_nonempty_union"].mean(),
        "Lesion_slice_Macro_Recall": lesion_slice_df["Slice_Recall"].mean(),
        "Union_positive_Macro_Dice": union_positive_df["Slice_Dice_nonempty_union"].mean(),
        "Patient_Macro_Dice": patient_df["Dice"].mean(),
        "Normal_slice_FP_rate": normal_slice_df["Prediction_positive"].mean(),
        "HD95_mm_both_positive_mean": case_df["HD95_mm_both_positive"].mean(),
        "HD95_mm_both_positive_median": case_df["HD95_mm_both_positive"].median(),
        "Completely_missed_lesion_slices": int(
            (case_df["GT_positive"] & ~case_df["Prediction_positive"]).sum()
        ),
        "False_positive_normal_slices": int(
            (~case_df["GT_positive"] & case_df["Prediction_positive"]).sum()
        ),
    }])
    return case_df, patient_df, summary_df

def load_foreground_probability(probability_path):
    with np.load(probability_path) as data:
        probabilities = data["probabilities"]
        return np.squeeze(probabilities[1]).astype(np.float32)

def remove_small_components(binary_mask, minimum_size):
    binary_mask = np.asarray(binary_mask).astype(bool)
    if minimum_size <= 0:
        return binary_mask
    component_map, component_count = connected_label(binary_mask)
    if component_count == 0:
        return binary_mask
    component_sizes = np.bincount(component_map.ravel())
    keep = component_sizes >= int(minimum_size)
    keep[0] = False
    return keep[component_map]

def case_input_lists(case_names, images_dir):
    return [
        [
            str(Path(images_dir) / f"{case_name}_{channel_index:04d}.nii.gz")
            for channel_index in INPUT_CHANNEL_INDICES
        ]
        for case_name in case_names
    ]

def validation_case_names():
    splits = read_json(SELECTED_PREPROCESSED_DATASET / "splits_final.json")
    assert FOLD < len(splits), f"Fold {FOLD}가 splits_final.json에 없습니다."
    return [str(item) for item in splits[FOLD]["val"]]

def test_case_names():
    labels_ts = SELECTED_RAW_DATASET / "labelsTs"
    assert labels_ts.is_dir(), f"labelsTs가 없습니다: {labels_ts}"
    return [path.name[:-7] for path in sorted(labels_ts.glob("*.nii.gz"))]

print("✅ CELL 4 완료")


In [ ]:
# ==================================================================================================
# CELL 5. 실행 경로 결정 및 Base 보호
# ==================================================================================================

EXPERIMENT_ID = f"Exp{EXP_NO:02d}"
EXPERIMENT_DIR = PATHS["experiments"] / EXPERIMENT_ID
MANUAL_RUNS_DIR = EXPERIMENT_DIR / "manual_runs"
OPTUNA_DIR = EXPERIMENT_DIR / "optuna"
OPTUNA_TRIALS_DIR = OPTUNA_DIR / "trials"
EXPERIMENT_BEST_DIR = EXPERIMENT_DIR / "best"
STUDY_NAME = f"BHSD_{EXPERIMENT_ID}_{NUM_CONTEXT_SLICES}Slice_Stage{OPTUNA_STAGE}"
STUDY_DB_PATH = OPTUNA_STUDIES_DIR / (
    f"{EXPERIMENT_ID}_{NUM_CONTEXT_SLICES}Slice_Stage{OPTUNA_STAGE}.db"
)

BASE_CHECKPOINT_HASH_BEFORE = sha256_file(BASE_CHECKPOINT)
BASE_TRAINER_HASH_BEFORE = sha256_file(BASE_TRAINER_FILE)

ACTIVE_RUN_DIR = None
MANUAL_RUN_ID = None

storage_modes = {"SMOKE", "MANUAL", "RESUME", "OPTUNA", "POSTPROCESS", "TEST_FINAL"}

if RUN_MODE in storage_modes:
    EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
    MANUAL_RUNS_DIR.mkdir(parents=True, exist_ok=True)
    OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
    OPTUNA_TRIALS_DIR.mkdir(parents=True, exist_ok=True)
    EXPERIMENT_BEST_DIR.mkdir(parents=True, exist_ok=True)
    CUSTOM_TRAINER_RUNTIME.mkdir(parents=True, exist_ok=True)
    OPTUNA_STUDIES_DIR.mkdir(parents=True, exist_ok=True)

if RUN_MODE in {"SMOKE", "MANUAL"}:
    run_number = next_run_number(MANUAL_RUNS_DIR)
    MANUAL_RUN_ID = f"RUN_{run_number:03d}"
    ACTIVE_RUN_DIR = MANUAL_RUNS_DIR / MANUAL_RUN_ID

elif RUN_MODE == "RESUME":
    assert re.fullmatch(r"RUN_\d{3}", RESUME_RUN_ID), "RESUME_RUN_ID 형식은 RUN_000입니다."
    MANUAL_RUN_ID = RESUME_RUN_ID
    ACTIVE_RUN_DIR = MANUAL_RUNS_DIR / MANUAL_RUN_ID
    assert ACTIVE_RUN_DIR.is_dir(), f"Resume 대상 RUN이 없습니다: {ACTIVE_RUN_DIR}"

def resolve_target_source():
    target_exp_id = f"Exp{TARGET_EXP_NO:02d}"
    target_exp_dir = PATHS["experiments"] / target_exp_id

    if TARGET_SOURCE_KIND == "MANUAL":
        source_dir = target_exp_dir / "manual_runs" / TARGET_RUN_ID
    elif TARGET_SOURCE_KIND == "OPTUNA":
        source_dir = target_exp_dir / "optuna" / "trials" / TARGET_RUN_ID
    elif TARGET_SOURCE_KIND == "EXPERIMENT_BEST":
        source_dir = target_exp_dir / "best"
    elif TARGET_SOURCE_KIND == "CURRENT_BEST":
        source_dir = CURRENT_BEST_DIR
    else:
        raise ValueError(f"지원하지 않는 TARGET_SOURCE_KIND: {TARGET_SOURCE_KIND}")

    assert source_dir.is_dir(), f"대상 폴더가 없습니다: {source_dir}"
    return source_dir

TARGET_SOURCE_DIR = None
if RUN_MODE in {"POSTPROCESS", "TEST_FINAL"}:
    TARGET_SOURCE_DIR = resolve_target_source()

print("=" * 100)
print("실행 경로")
print("=" * 100)
print("EXPERIMENT_ID :", EXPERIMENT_ID)
print("RUN_MODE      :", RUN_MODE)
print("ACTIVE_RUN    :", ACTIVE_RUN_DIR)
print("STUDY_NAME    :", STUDY_NAME)
print("TARGET_SOURCE :", TARGET_SOURCE_DIR)
print("✅ CELL 5 완료")


In [ ]:
# ==================================================================================================
# CELL 6. Runtime Custom Trainer 생성
# ==================================================================================================
# Base Trainer 파일은 수정하지 않는다.
# 신규 실험 전용 Trainer만 custom_trainers_runtime 및 현재 런타임 nnU-Net 설치 경로에 등록한다.

TRAINER_CLASS_NAME = "TeamExperimentTrainer"

TRAINER_CODE = r"""
import json
import math
import os

import numpy as np
import torch
from torch import nn

from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss
from nnunetv2.training.lr_scheduler.polylr import PolyLRScheduler


def _load_config():
    with open(os.environ["TEAM_EXPERIMENT_CONFIG"], "r", encoding="utf-8") as file_obj:
        return json.load(file_obj)


class BinarySoftDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = float(smooth)

    def forward(self, logits, target):
        if target.ndim == logits.ndim:
            target = target[:, 0]
        target = target.long()
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        foreground = (target == 1).float()
        axes = tuple(range(1, probabilities.ndim))
        tp = (probabilities * foreground).sum(axes)
        fp = (probabilities * (1 - foreground)).sum(axes)
        fn = ((1 - probabilities) * foreground).sum(axes)
        dice = (2 * tp + self.smooth) / (2 * tp + fp + fn + self.smooth)
        return 1 - dice.mean()


class BinaryTverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=1.0, smooth=1e-5):
        super().__init__()
        self.alpha = float(alpha)
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.smooth = float(smooth)

    def forward(self, logits, target):
        if target.ndim == logits.ndim:
            target = target[:, 0]
        target = target.long()
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        foreground = (target == 1).float()
        axes = tuple(range(1, probabilities.ndim))
        tp = (probabilities * foreground).sum(axes)
        fp = (probabilities * (1 - foreground)).sum(axes)
        fn = ((1 - probabilities) * foreground).sum(axes)
        score = (
            tp + self.smooth
        ) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )
        return torch.pow(1 - score, self.gamma).mean()


class TeamExperimentTrainer(nnUNetTrainer):
    def __init__(
        self,
        plans,
        configuration,
        fold,
        dataset_json,
        device=torch.device("cuda"),
    ):
        super().__init__(
            plans=plans,
            configuration=configuration,
            fold=fold,
            dataset_json=dataset_json,
            device=device,
        )
        cfg = _load_config()
        self.team_cfg = cfg
        self.num_epochs = int(cfg["trial_epochs"])
        self.initial_lr = float(cfg["learning_rate"])
        self.weight_decay = float(cfg["weight_decay"])
        self.num_iterations_per_epoch = int(cfg["train_iters_per_epoch"])
        self.num_val_iterations_per_epoch = int(cfg["val_iters_per_epoch"])
        self.oversample_foreground_percent = float(
            cfg["oversample_foreground_percent"]
        )
        self.configuration_manager.configuration["batch_size"] = int(cfg["batch_size"])
        self.early_stopping_min_epochs = int(cfg["early_stopping_min_epochs"])
        self.early_stopping_patience = int(cfg["early_stopping_patience"])
        self.early_stopping_min_delta = float(cfg["early_stopping_min_delta"])
        self.early_stopping_best_score = None
        self.early_stopping_wait = 0
        self.early_stopping_triggered = False
        self.save_every = 1

    def _wrap_deep_supervision(self, loss):
        if self.enable_deep_supervision:
            scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** index) for index in range(len(scales))])
            if len(weights) > 1:
                weights[-1] = 0
            weights = weights / weights.sum()
            return DeepSupervisionWrapper(loss, weights)
        return loss

    def _build_loss(self):
        cfg = self.team_cfg
        loss_name = cfg["loss"]

        if loss_name == "DiceCE":
            loss = DC_and_CE_loss(
                {
                    "batch_dice": self.configuration_manager.batch_dice,
                    "smooth": 1e-5,
                    "do_bg": False,
                    "ddp": self.is_ddp,
                },
                {},
                weight_ce=1,
                weight_dice=1,
                ignore_label=self.label_manager.ignore_label,
                dice_class=MemoryEfficientSoftDiceLoss,
            )
        elif loss_name == "Dice":
            loss = BinarySoftDiceLoss()
        elif loss_name == "Tversky":
            loss = BinaryTverskyLoss(
                alpha=cfg["tversky_alpha"],
                beta=cfg["tversky_beta"],
                gamma=1.0,
            )
        elif loss_name == "FocalTversky":
            loss = BinaryTverskyLoss(
                alpha=cfg["tversky_alpha"],
                beta=cfg["tversky_beta"],
                gamma=cfg["focal_gamma"],
            )
        else:
            raise ValueError(f"지원하지 않는 Loss: {loss_name}")

        return self._wrap_deep_supervision(loss)

    def configure_optimizers(self):
        cfg = self.team_cfg
        lr = float(cfg["learning_rate"])
        weight_decay = float(cfg["weight_decay"])

        if cfg["optimizer"] == "SGD":
            optimizer = torch.optim.SGD(
                self.network.parameters(),
                lr=lr,
                momentum=0.99,
                nesterov=True,
                weight_decay=weight_decay,
            )
        elif cfg["optimizer"] == "Adam":
            optimizer = torch.optim.Adam(
                self.network.parameters(),
                lr=lr,
                weight_decay=weight_decay,
            )
        elif cfg["optimizer"] == "AdamW":
            optimizer = torch.optim.AdamW(
                self.network.parameters(),
                lr=lr,
                weight_decay=weight_decay,
            )
        else:
            raise ValueError(f"지원하지 않는 Optimizer: {cfg['optimizer']}")

        if cfg["scheduler"] == "PolyLR":
            scheduler = PolyLRScheduler(optimizer, lr, self.num_epochs)
        elif cfg["scheduler"] == "Cosine":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=self.num_epochs,
            )
        elif cfg["scheduler"] == "StepLR":
            scheduler = torch.optim.lr_scheduler.StepLR(
                optimizer,
                step_size=max(1, self.num_epochs // 3),
                gamma=0.1,
            )
        else:
            raise ValueError(f"지원하지 않는 Scheduler: {cfg['scheduler']}")

        return optimizer, scheduler

    def on_epoch_end(self):
        super().on_epoch_end()

        completed_epochs = int(self.current_epoch) + 1
        if completed_epochs < self.early_stopping_min_epochs:
            return

        score = None
        candidates = self.logger.my_fantastic_logging.get("ema_fg_dice", [])
        if candidates:
            try:
                score = float(candidates[-1])
            except Exception:
                score = None

        if score is None or not math.isfinite(score):
            self.early_stopping_wait += 1
        elif (
            self.early_stopping_best_score is None
            or score > self.early_stopping_best_score + self.early_stopping_min_delta
        ):
            self.early_stopping_best_score = score
            self.early_stopping_wait = 0
        else:
            self.early_stopping_wait += 1

        if self.early_stopping_wait >= self.early_stopping_patience:
            self.early_stopping_triggered = True
            self.print_to_log_file(
                "[Team Early Stopping] "
                f"Epoch={completed_epochs}, "
                f"Best={self.early_stopping_best_score}, "
                f"Patience={self.early_stopping_patience}"
            )
            self.current_epoch = self.num_epochs - 1
"""

if RUN_MODE in {"SMOKE", "MANUAL", "RESUME", "OPTUNA"}:
    runtime_trainer_path = CUSTOM_TRAINER_RUNTIME / f"{TRAINER_CLASS_NAME}.py"
    runtime_trainer_path.write_text(TRAINER_CODE, encoding="utf-8")

    site_trainer_dir = (
        Path(site.getsitepackages()[0])
        / "nnunetv2"
        / "training"
        / "nnUNetTrainer"
    )
    assert site_trainer_dir.is_dir(), f"nnU-Net Trainer 설치 경로 없음: {site_trainer_dir}"

    installed_trainer_path = site_trainer_dir / f"{TRAINER_CLASS_NAME}.py"
    shutil.copy2(runtime_trainer_path, installed_trainer_path)

    for cache_dir in site_trainer_dir.rglob("__pycache__"):
        shutil.rmtree(cache_dir, ignore_errors=True)

    print("Runtime Trainer:", runtime_trainer_path)
    print("Installed Trainer:", installed_trainer_path)
else:
    print("현재 모드는 신규 Trainer 생성이 필요하지 않습니다.")

print("✅ CELL 6 완료")


In [ ]:
# ==================================================================================================
# CELL 7. Trial 설정·Optuna Pruner·학습 실행 함수
# ==================================================================================================

def build_trial_config(trial=None):
    cfg = {
        "experiment_id": EXPERIMENT_ID,
        "run_mode": RUN_MODE,
        "profile": PROFILE,
        "model_mode": MODEL_MODE,
        "num_context_slices": NUM_CONTEXT_SLICES,
        "dataset_id": DATASET_ID,
        "dataset_name": DATASET_NAME,
        "configuration": NNUNET_CONFIGURATION,
        "plans_name": PLANS_NAME,
        "fold": FOLD,
        "loss": LOSS,
        "optimizer": OPTIMIZER,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": SCHEDULER,
        "batch_size": BATCH_SIZE,
        "oversample_foreground_percent": OVERSAMPLE_FOREGROUND_PERCENT,
        "positive_case_ratio": POSITIVE_CASE_RATIO,
        "tversky_alpha": TVERSKY_ALPHA,
        "tversky_beta": TVERSKY_BETA,
        "focal_gamma": FOCAL_GAMMA,
        "trial_epochs": TRAIN_EPOCHS,
        "train_iters_per_epoch": TRAIN_ITERS_PER_EPOCH,
        "val_iters_per_epoch": VAL_ITERS_PER_EPOCH,
        "early_stopping_min_epochs": EARLY_STOPPING_MIN_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
        "random_seed": RANDOM_SEED,
        "use_base_pretrained_weights": USE_BASE_PRETRAINED_WEIGHTS,
    }

    if trial is None:
        return cfg

    if SEARCH_LOSS:
        cfg["loss"] = trial.suggest_categorical("loss", SEARCH_SPACE["loss"])
    if SEARCH_OPTIMIZER:
        cfg["optimizer"] = trial.suggest_categorical("optimizer", SEARCH_SPACE["optimizer"])
    if SEARCH_LR:
        cfg["learning_rate"] = trial.suggest_float(
            "learning_rate",
            SEARCH_SPACE["lr_low"],
            SEARCH_SPACE["lr_high"],
            log=True,
        )
    if SEARCH_WEIGHT_DECAY:
        cfg["weight_decay"] = trial.suggest_float(
            "weight_decay",
            SEARCH_SPACE["weight_decay_low"],
            SEARCH_SPACE["weight_decay_high"],
            log=True,
        )
    if SEARCH_SCHEDULER:
        cfg["scheduler"] = trial.suggest_categorical(
            "scheduler",
            SEARCH_SPACE["scheduler"],
        )
    if SEARCH_BATCH_SIZE:
        cfg["batch_size"] = trial.suggest_categorical(
            "batch_size",
            SEARCH_SPACE["batch_size"],
        )
    if SEARCH_OVERSAMPLE:
        cfg["oversample_foreground_percent"] = trial.suggest_categorical(
            "oversample_foreground_percent",
            SEARCH_SPACE["oversample_foreground_percent"],
        )
    if SEARCH_POSITIVE_CASE_RATIO:
        cfg["positive_case_ratio"] = trial.suggest_categorical(
            "positive_case_ratio",
            SEARCH_SPACE["positive_case_ratio"],
        )
    if cfg["loss"] in {"Tversky", "FocalTversky"} and SEARCH_TVERSKY_ALPHA:
        alpha = trial.suggest_categorical(
            "tversky_alpha",
            SEARCH_SPACE["tversky_alpha"],
        )
        cfg["tversky_alpha"] = alpha
        cfg["tversky_beta"] = 1.0 - alpha
    if cfg["loss"] == "FocalTversky" and SEARCH_FOCAL_GAMMA:
        cfg["focal_gamma"] = trial.suggest_categorical(
            "focal_gamma",
            SEARCH_SPACE["focal_gamma"],
        )

    return cfg

def create_pruner():
    import optuna

    if PRUNER_NAME == "NONE":
        return optuna.pruners.NopPruner()
    if PRUNER_NAME == "MEDIAN":
        return optuna.pruners.MedianPruner(
            n_startup_trials=PRUNER_STARTUP_TRIALS,
            n_warmup_steps=PRUNER_WARMUP_STEPS,
            interval_steps=PRUNER_INTERVAL_STEPS,
        )
    if PRUNER_NAME == "PERCENTILE":
        return optuna.pruners.PercentilePruner(
            percentile=PRUNER_PERCENTILE,
            n_startup_trials=PRUNER_STARTUP_TRIALS,
            n_warmup_steps=PRUNER_WARMUP_STEPS,
            interval_steps=PRUNER_INTERVAL_STEPS,
        )
    if PRUNER_NAME == "HYPERBAND":
        return optuna.pruners.HyperbandPruner(
            min_resource=PRUNER_MIN_RESOURCE,
            max_resource=TRAIN_EPOCHS,
            reduction_factor=PRUNER_REDUCTION_FACTOR,
        )
    if PRUNER_NAME == "SUCCESSIVE_HALVING":
        return optuna.pruners.SuccessiveHalvingPruner(
            min_resource=PRUNER_MIN_RESOURCE,
            reduction_factor=PRUNER_REDUCTION_FACTOR,
        )
    raise ValueError(PRUNER_NAME)

def find_latest_checkpoint(root: Path, checkpoint_name: str):
    candidates = sorted(
        root.rglob(checkpoint_name),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

def numeric_values(value):
    if value is None:
        return []
    if isinstance(value, (int, float, np.integer, np.floating)):
        return [float(value)]
    if torch.is_tensor(value):
        return [
            float(item)
            for item in value.detach().cpu().float().reshape(-1).tolist()
        ]
    if isinstance(value, np.ndarray):
        return [float(item) for item in value.astype(float).reshape(-1).tolist()]
    if isinstance(value, (list, tuple)):
        output = []
        for item in value:
            output.extend(numeric_values(item))
        return output
    return []

def extract_checkpoint_dice(checkpoint_path: Path):
    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

    records = []

    def walk(obj, current_path="checkpoint"):
        if isinstance(obj, dict):
            for key, value in obj.items():
                next_path = f"{current_path}.{key}"
                key_lower = str(key).lower()
                if "dice" in key_lower or "foreground" in key_lower:
                    values = [
                        number
                        for number in numeric_values(value)
                        if math.isfinite(number) and 0.0 <= number <= 1.0
                    ]
                    if values:
                        records.append({
                            "path": next_path,
                            "values": values,
                            "max": max(values),
                            "last": values[-1],
                        })
                walk(value, next_path)
        elif isinstance(obj, (list, tuple)):
            for index, value in enumerate(obj):
                walk(value, f"{current_path}[{index}]")

    walk(checkpoint)
    if not records:
        raise ValueError("checkpoint에서 Dice 기록을 찾지 못했습니다.")

    priority = [
        "ema_fg_dice",
        "ema_foreground_dice",
        "mean_fg_dice",
        "foreground_mean_dice",
        "fg_dice",
        "dice",
    ]
    selected = None
    for keyword in priority:
        matches = [record for record in records if keyword in record["path"].lower()]
        if matches:
            selected = max(matches, key=lambda record: record["max"])
            break
    if selected is None:
        selected = max(records, key=lambda record: record["max"])

    return {
        "best_validation_dice_internal": float(selected["max"]),
        "latest_validation_dice_internal": float(selected["last"]),
        "best_epoch_internal": int(np.argmax(selected["values"])),
        "dice_source": selected["path"],
        "dice_history": selected["values"],
    }

def terminate_process(process):
    if process.poll() is not None:
        return
    process.send_signal(signal.SIGTERM)
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()

print("✅ CELL 7 완료")


In [ ]:
# ==================================================================================================
# CELL 8. Prediction·후처리·Validation/Test 평가 함수
# ==================================================================================================

def register_trainer_file_for_inference(trainer_file: Path):
    site_trainer_dir = (
        Path(site.getsitepackages()[0])
        / "nnunetv2"
        / "training"
        / "nnUNetTrainer"
    )
    destination = site_trainer_dir / trainer_file.name
    if not destination.exists() or sha256_file(destination) != sha256_file(trainer_file):
        shutil.copy2(trainer_file, destination)
    return destination

def make_base_runtime_config():
    metadata = read_json(BASE_METADATA_PATH)
    default_cfg = {
        "num_epochs": 100,
        "num_iterations_per_epoch": 250,
        "num_val_iterations_per_epoch": 40,
        "early_stopping_min_epochs": 80,
        "early_stopping_patience": 40,
        "early_stopping_min_delta": 0.001,
        "initial_lr": 5e-4,
        "weight_decay": 1e-5,
        "optimizer_name": "adamw",
        "scheduler_name": "poly",
        "loss_name": "tversky_ce",
        "tversky_alpha": 0.3,
        "tversky_beta": 0.7,
        "focal_gamma": 1.0,
        "oversample_foreground_percent": 0.33,
        "positive_case_ratio": 0.5,
        "batch_size": 8,
    }

    def merge_numeric(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                normalized = str(key).lower()
                if normalized in default_cfg and isinstance(value, (int, float, str, bool)):
                    default_cfg[normalized] = value
                merge_numeric(value)
        elif isinstance(obj, list):
            for value in obj:
                merge_numeric(value)

    merge_numeric(metadata)
    return default_cfg

def predict_with_probabilities(
    model_dir: Path,
    case_names,
    images_dir: Path,
    output_dir: Path,
    base_mode: bool = False,
):
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

    output_dir.mkdir(parents=True, exist_ok=True)

    if base_mode:
        register_trainer_file_for_inference(BASE_TRAINER_FILE)
        base_cfg_path = Path("/content/base_model_runtime_config.json")
        write_json(base_cfg_path, make_base_runtime_config())
        os.environ["BHSD_CONFIG_PATH"] = str(base_cfg_path)
        os.environ["EXP03_CONFIG_PATH"] = str(base_cfg_path)

    predictor = nnUNetPredictor(
        tile_step_size=SLIDING_WINDOW_STEP_SIZE,
        use_gaussian=True,
        use_mirroring=True,
        perform_everything_on_device=True,
        device=torch.device("cuda"),
        verbose=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(model_dir),
        use_folds=(FOLD,),
        checkpoint_name="checkpoint_best.pth",
    )
    predictor.predict_from_files(
        case_input_lists(case_names, images_dir),
        [str(output_dir / case_name) for case_name in case_names],
        save_probabilities=True,
        overwrite=True,
        num_processes_preprocessing=3,
        num_processes_segmentation_export=3,
    )
    return output_dir

def apply_postprocess_to_probability_folder(
    probability_dir: Path,
    output_dir: Path,
    ground_truth_dir: Path,
    case_names,
    threshold: float,
    minimum_size: int,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    for case_name in case_names:
        probability_path = probability_dir / f"{case_name}.npz"
        reference_path = ground_truth_dir / f"{case_name}.nii.gz"
        assert probability_path.is_file(), probability_path
        assert reference_path.is_file(), reference_path

        probability = load_foreground_probability(probability_path)
        prediction = remove_small_components(
            probability >= threshold,
            minimum_size,
        )
        reference_nii = nib.load(str(reference_path))
        output_nii = nib.Nifti1Image(
            prediction.astype(np.uint8),
            reference_nii.affine,
            reference_nii.header,
        )
        nib.save(output_nii, str(output_dir / f"{case_name}.nii.gz"))

def tune_validation_postprocessing(
    probability_dir: Path,
    ground_truth_dir: Path,
    case_names,
    output_root: Path,
):
    rows = []

    for threshold in THRESHOLD_CANDIDATES:
        for minimum_size in MIN_COMPONENT_SIZE_CANDIDATES:
            candidate_dir = output_root / (
                f"threshold_{threshold:.2f}_component_{minimum_size}"
            )
            apply_postprocess_to_probability_folder(
                probability_dir=probability_dir,
                output_dir=candidate_dir,
                ground_truth_dir=ground_truth_dir,
                case_names=case_names,
                threshold=threshold,
                minimum_size=minimum_size,
            )
            _, _, summary_df = evaluate_prediction_folder(
                candidate_dir,
                ground_truth_dir,
                "Validation",
            )
            record = summary_df.iloc[0].to_dict()
            record["Threshold"] = threshold
            record["Minimum_component_size"] = minimum_size
            record["Prediction_dir"] = str(candidate_dir)
            rows.append(record)

    tuning_df = pd.DataFrame(rows).sort_values(
        ["Micro_Dice", "Micro_Recall"],
        ascending=[False, False],
    ).reset_index(drop=True)

    best = tuning_df.iloc[0].to_dict()
    return tuning_df, best

def locate_model_dir_from_source(source_dir: Path):
    if source_dir == CURRENT_BEST_DIR:
        current_payload = read_json(CURRENT_BEST_JSON)
        checkpoint_path = Path(
            current_payload.get(
                "checkpoint_best",
                current_payload.get("checkpoint_registry_path", ""),
            )
        )
        assert checkpoint_path.is_file(), "Current Best checkpoint를 찾을 수 없습니다."
        source_model_dir = source_dir / "nnUNet_model"
        if source_model_dir.is_dir():
            return source_model_dir
        raise FileNotFoundError(
            "Current Best에 독립 추론용 nnUNet_model 폴더가 없습니다. "
            "해당 후보의 source_run을 지정해 TEST_FINAL을 실행하세요."
        )

    results_root = source_dir / "nnUNet_results"
    assert results_root.is_dir(), f"RUN/Trial nnUNet_results가 없습니다: {results_root}"
    candidates = [
        path
        for path in results_root.rglob("*__nnUNetPlans__2d")
        if path.is_dir()
    ]
    assert candidates, f"학습 모델 폴더를 찾지 못했습니다: {results_root}"
    return candidates[0]

print("✅ CELL 8 완료")


In [ ]:
# ==================================================================================================
# CELL 9. 학습 1회 실행 및 전체 Validation 평가
# ==================================================================================================

def run_training_once(trial=None):
    import optuna

    trial_number = int(trial.number) if trial is not None else 0

    if trial is None:
        assert ACTIVE_RUN_DIR is not None
        run_dir = ACTIVE_RUN_DIR
    else:
        run_dir = OPTUNA_TRIALS_DIR / f"TRIAL_{trial_number:03d}"

    if RUN_MODE != "RESUME":
        if run_dir.exists():
            if trial is not None:
                existing_result = run_dir / "trial_result.json"
                if existing_result.is_file():
                    old = read_json(existing_result)
                    if old.get("status") == "COMPLETE":
                        return float(old["validation_metrics"]["Micro_Dice"])
                shutil.rmtree(run_dir)
            else:
                raise FileExistsError(f"신규 RUN 폴더가 이미 존재합니다: {run_dir}")
        run_dir.mkdir(parents=True, exist_ok=False)

    results_root = run_dir / "nnUNet_results"
    results_root.mkdir(parents=True, exist_ok=True)

    trial_cfg = build_trial_config(trial)
    trial_cfg["trial_number"] = trial_number
    trial_cfg["run_dir"] = str(run_dir)
    trial_cfg["trial_results_root"] = str(results_root)
    trial_cfg["created_at"] = datetime.now().isoformat(timespec="seconds")
    trial_cfg["config_hash"] = config_hash(trial_cfg)

    config_path = run_dir / "trial_config.json"
    log_path = run_dir / "console_log.txt"
    result_path = run_dir / "trial_result.json"
    write_json(config_path, trial_cfg)

    env = os.environ.copy()
    env["TEAM_EXPERIMENT_CONFIG"] = str(config_path)
    env["nnUNet_results"] = str(results_root)

    executable = shutil.which("nnUNetv2_train")
    assert executable is not None

    command = [
        executable,
        str(DATASET_ID),
        NNUNET_CONFIGURATION,
        str(FOLD),
        "-tr",
        TRAINER_CLASS_NAME,
        "-p",
        PLANS_NAME,
        "--npz",
        "-device",
        "cuda",
    ]

    if RUN_MODE == "RESUME":
        command.append("--c")
    elif USE_BASE_PRETRAINED_WEIGHTS:
        command.extend([
            "-pretrained_weights",
            str(BASE_CHECKPOINT),
        ])

    start_time = time.time()
    deadline = start_time + MAX_TRIAL_MINUTES * 60
    global_deadline = globals().get("GLOBAL_DEADLINE", float("inf"))
    last_checkpoint_mtime = None
    report_step = 0
    last_internal = None
    stop_reason = None

    print("=" * 100)
    print("학습 시작:", run_dir)
    print("Command:", " ".join(command))
    print("=" * 100)

    with log_path.open("a" if RUN_MODE == "RESUME" else "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
            env=env,
        )

        while process.poll() is None:
            time.sleep(CHECK_INTERVAL_SECONDS)
            now = time.time()

            if now >= global_deadline:
                stop_reason = "GLOBAL_TIME_LIMIT"
                terminate_process(process)
                break
            if now >= deadline:
                stop_reason = "TRIAL_TIME_LIMIT"
                terminate_process(process)
                break

            checkpoint = (
                find_latest_checkpoint(results_root, "checkpoint_latest.pth")
                or find_latest_checkpoint(results_root, "checkpoint_best.pth")
            )
            if checkpoint is None:
                continue
            mtime = checkpoint.stat().st_mtime
            if mtime == last_checkpoint_mtime:
                continue
            last_checkpoint_mtime = mtime

            try:
                last_internal = extract_checkpoint_dice(checkpoint)
            except Exception:
                continue

            report_step += 1
            latest_value = last_internal["latest_validation_dice_internal"]
            print(
                f"Trial {trial_number} | Step {report_step} | "
                f"Internal Latest Dice {latest_value:.6f}"
            )

            if trial is not None and PRUNER_NAME != "NONE":
                trial.report(latest_value, step=report_step)
                if trial.should_prune():
                    stop_reason = "PRUNED"
                    terminate_process(process)
                    break

    checkpoint = (
        find_latest_checkpoint(results_root, "checkpoint_best.pth")
        or find_latest_checkpoint(results_root, "checkpoint_latest.pth")
    )

    if stop_reason == "PRUNED":
        write_json(result_path, {
            "status": "PRUNED",
            "trial_number": trial_number,
            "elapsed_seconds": time.time() - start_time,
            "checkpoint_best": str(checkpoint) if checkpoint else None,
            "config": trial_cfg,
        })
        raise optuna.TrialPruned()

    if checkpoint is None:
        write_json(result_path, {
            "status": "FAILED",
            "trial_number": trial_number,
            "return_code": process.returncode,
            "elapsed_seconds": time.time() - start_time,
            "checkpoint_best": None,
            "config": trial_cfg,
        })
        raise RuntimeError(f"checkpoint가 생성되지 않았습니다. 로그: {log_path}")

    model_dir = locate_model_dir_from_source(run_dir)
    val_names = validation_case_names()
    validation_dir = run_dir / "evaluation" / "validation"
    raw_probability_dir = validation_dir / "raw_probability"
    postprocess_root = validation_dir / "postprocess_candidates"

    predict_with_probabilities(
        model_dir=model_dir,
        case_names=val_names,
        images_dir=SELECTED_RAW_DATASET / "imagesTr",
        output_dir=raw_probability_dir,
        base_mode=False,
    )

    tuning_df, best_postprocess = tune_validation_postprocessing(
        probability_dir=raw_probability_dir,
        ground_truth_dir=SELECTED_RAW_DATASET / "labelsTr",
        case_names=val_names,
        output_root=postprocess_root,
    )

    selected_prediction_dir = Path(best_postprocess["Prediction_dir"])
    case_df, patient_df, summary_df = evaluate_prediction_folder(
        selected_prediction_dir,
        SELECTED_RAW_DATASET / "labelsTr",
        "Validation",
    )

    validation_dir.mkdir(parents=True, exist_ok=True)
    tuning_df.to_csv(
        validation_dir / "postprocess_tuning.csv",
        index=False,
        encoding="utf-8-sig",
    )
    case_df.to_csv(
        validation_dir / "validation_case_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )
    patient_df.to_csv(
        validation_dir / "validation_patient_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )
    summary_df.to_csv(
        validation_dir / "validation_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    validation_metrics = summary_df.iloc[0].to_dict()
    postprocess_config = {
        "threshold": float(best_postprocess["Threshold"]),
        "minimum_component_size": int(best_postprocess["Minimum_component_size"]),
        "selected_prediction_dir": str(selected_prediction_dir),
    }
    write_json(validation_dir / "validation_metrics.json", validation_metrics)
    write_json(validation_dir / "postprocess_config.json", postprocess_config)

    result = {
        "status": stop_reason or "COMPLETE",
        "trial_number": trial_number,
        "return_code": process.returncode,
        "elapsed_seconds": time.time() - start_time,
        "checkpoint_best": str(checkpoint),
        "model_dir": str(model_dir),
        "console_log_path": str(log_path),
        "config": trial_cfg,
        "validation_metrics": validation_metrics,
        "postprocess_config": postprocess_config,
    }
    write_json(result_path, result)

    return float(validation_metrics["Micro_Dice"])

print("✅ CELL 9 완료")


In [ ]:
# ==================================================================================================
# CELL 10. RUN_MODE 실행
# ==================================================================================================

import optuna

BEST_SOURCE_DIR = None
BEST_VALUE = None
BEST_TRIAL_NUMBER = None

if RUN_MODE == "BASE_EVALUATE":
    print("BASE_EVALUATE는 /content 임시 폴더만 사용하며 Drive에 평가 결과를 저장하지 않습니다.")

    with tempfile.TemporaryDirectory(prefix="base_evaluate_", dir="/content") as temp_dir_text:
        temp_dir = Path(temp_dir_text)

        val_names = validation_case_names()
        val_probability_dir = temp_dir / "validation_probability"
        predict_with_probabilities(
            model_dir=BASE_RESULT_MODEL_DIR,
            case_names=val_names,
            images_dir=SELECTED_RAW_DATASET / "imagesTr",
            output_dir=val_probability_dir,
            base_mode=True,
        )
        _, val_best = tune_validation_postprocessing(
            probability_dir=val_probability_dir,
            ground_truth_dir=SELECTED_RAW_DATASET / "labelsTr",
            case_names=val_names,
            output_root=temp_dir / "validation_postprocess",
        )
        val_prediction_dir = Path(val_best["Prediction_dir"])
        _, _, validation_summary = evaluate_prediction_folder(
            val_prediction_dir,
            SELECTED_RAW_DATASET / "labelsTr",
            "Validation",
        )

        names_test = test_case_names()
        test_probability_dir = temp_dir / "test_probability"
        predict_with_probabilities(
            model_dir=BASE_RESULT_MODEL_DIR,
            case_names=names_test,
            images_dir=SELECTED_RAW_DATASET / "imagesTs",
            output_dir=test_probability_dir,
            base_mode=True,
        )
        selected_test_dir = temp_dir / "test_selected"
        apply_postprocess_to_probability_folder(
            probability_dir=test_probability_dir,
            output_dir=selected_test_dir,
            ground_truth_dir=SELECTED_RAW_DATASET / "labelsTs",
            case_names=names_test,
            threshold=float(val_best["Threshold"]),
            minimum_size=int(val_best["Minimum_component_size"]),
        )
        _, _, test_summary = evaluate_prediction_folder(
            selected_test_dir,
            SELECTED_RAW_DATASET / "labelsTs",
            "Test",
        )

        print("=" * 110)
        print("BASE_EVALUATE 결과 — Drive 저장 없음")
        print("=" * 110)
        print("Validation")
        display(validation_summary)
        print("Test")
        display(test_summary)
        print("Validation Threshold:", val_best["Threshold"])
        print("Validation Minimum Component:", val_best["Minimum_component_size"])
        print("=" * 110)

elif RUN_MODE in {"SMOKE", "MANUAL", "RESUME"}:
    if RUN_MODE in {"SMOKE", "MANUAL"}:
        manual_cfg = build_trial_config(None)
        manual_cfg["manual_run_id"] = MANUAL_RUN_ID
        manual_cfg["created_at"] = datetime.now().isoformat(timespec="seconds")
        manual_cfg["config_hash"] = config_hash(manual_cfg)

        duplicates = []
        for config_path in MANUAL_RUNS_DIR.glob("RUN_*/run_config.json"):
            try:
                old = read_json(config_path)
                if old.get("config_hash") == manual_cfg["config_hash"]:
                    duplicates.append(config_path.parent)
            except Exception:
                pass
        if duplicates:
            print("⚠️ 동일 설정 기존 RUN:", duplicates[-1])
            if BLOCK_DUPLICATE_MANUAL_CONFIG:
                raise RuntimeError(f"동일 설정이 이미 존재합니다: {duplicates[-1]}")

    GLOBAL_DEADLINE = time.time() + MAX_TOTAL_TRAINING_MINUTES * 60
    BEST_VALUE = float(run_training_once(None))
    BEST_SOURCE_DIR = ACTIVE_RUN_DIR

elif RUN_MODE == "OPTUNA":
    GLOBAL_DEADLINE = time.time() + MAX_TOTAL_TRAINING_MINUTES * 60
    study = optuna.create_study(
        study_name=STUDY_NAME,
        storage=f"sqlite:///{STUDY_DB_PATH}",
        direction="maximize",
        pruner=create_pruner(),
        load_if_exists=True,
    )
    study.optimize(
        run_training_once,
        n_trials=N_TRIALS,
        timeout=MAX_TOTAL_TRAINING_MINUTES * 60,
        catch=(RuntimeError,),
        gc_after_trial=True,
    )
    completed = [
        trial
        for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
        and trial.value is not None
    ]
    assert completed, "완료된 Optuna Trial이 없습니다."
    best_trial = max(completed, key=lambda trial: trial.value)
    BEST_TRIAL_NUMBER = int(best_trial.number)
    BEST_VALUE = float(best_trial.value)
    BEST_SOURCE_DIR = OPTUNA_TRIALS_DIR / f"TRIAL_{BEST_TRIAL_NUMBER:03d}"

elif RUN_MODE == "POSTPROCESS":
    probability_dir = TARGET_SOURCE_DIR / "evaluation" / "validation" / "raw_probability"
    assert probability_dir.is_dir(), f"Validation probability 폴더가 없습니다: {probability_dir}"

    postprocess_version = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_root = (
        TARGET_SOURCE_DIR
        / "postprocess"
        / f"POSTPROCESS_{postprocess_version}"
    )
    output_root.mkdir(parents=True, exist_ok=False)

    tuning_df, best_postprocess = tune_validation_postprocessing(
        probability_dir=probability_dir,
        ground_truth_dir=SELECTED_RAW_DATASET / "labelsTr",
        case_names=validation_case_names(),
        output_root=output_root / "candidates",
    )
    tuning_df.to_csv(
        output_root / "postprocess_tuning.csv",
        index=False,
        encoding="utf-8-sig",
    )
    write_json(output_root / "postprocess_config.json", best_postprocess)
    print("POSTPROCESS 완료:", output_root)
    display(tuning_df.head(10))

elif RUN_MODE == "TEST_FINAL":
    model_dir = locate_model_dir_from_source(TARGET_SOURCE_DIR)

    validation_config_candidates = [
        TARGET_SOURCE_DIR / "evaluation" / "validation" / "postprocess_config.json",
        TARGET_SOURCE_DIR / "postprocess_config.json",
    ]
    postprocess_config_path = next(
        (path for path in validation_config_candidates if path.is_file()),
        None,
    )
    assert postprocess_config_path is not None, (
        "Validation에서 선택한 postprocess_config.json이 없습니다."
    )
    postprocess_cfg = read_json(postprocess_config_path)

    test_version = datetime.now().strftime("%Y%m%d_%H%M%S")
    test_dir = TARGET_SOURCE_DIR / "evaluation" / f"test_final_{test_version}"
    probability_dir = test_dir / "raw_probability"
    selected_dir = test_dir / "selected_prediction"

    names_test = test_case_names()
    predict_with_probabilities(
        model_dir=model_dir,
        case_names=names_test,
        images_dir=SELECTED_RAW_DATASET / "imagesTs",
        output_dir=probability_dir,
        base_mode=False,
    )
    apply_postprocess_to_probability_folder(
        probability_dir=probability_dir,
        output_dir=selected_dir,
        ground_truth_dir=SELECTED_RAW_DATASET / "labelsTs",
        case_names=names_test,
        threshold=float(postprocess_cfg["threshold"]),
        minimum_size=int(postprocess_cfg["minimum_component_size"]),
    )
    case_df, patient_df, summary_df = evaluate_prediction_folder(
        selected_dir,
        SELECTED_RAW_DATASET / "labelsTs",
        "Test",
    )
    case_df.to_csv(test_dir / "test_case_metrics.csv", index=False, encoding="utf-8-sig")
    patient_df.to_csv(test_dir / "test_patient_metrics.csv", index=False, encoding="utf-8-sig")
    summary_df.to_csv(test_dir / "test_summary.csv", index=False, encoding="utf-8-sig")
    write_json(test_dir / "test_metrics.json", summary_df.iloc[0].to_dict())
    write_json(test_dir / "applied_postprocess_config.json", postprocess_cfg)

    print("TEST_FINAL 완료:", test_dir)
    display(summary_df)

print("✅ CELL 10 완료")


In [ ]:
# ==================================================================================================
# CELL 11. Experiment Best·Current Best 백업 및 갱신
# ==================================================================================================
# BASE_EVALUATE / POSTPROCESS / TEST_FINAL에서는 실행하지 않는다.
# SMOKE는 ALLOW_SMOKE_BEST_UPDATE=False이면 Best 갱신에서 제외한다.

best_management_allowed = RUN_MODE in {"MANUAL", "RESUME", "OPTUNA"}
if RUN_MODE == "SMOKE" and ALLOW_SMOKE_BEST_UPDATE:
    best_management_allowed = True

EXPERIMENT_BEST_UPDATED = False
GLOBAL_BEST_UPDATED = False

if best_management_allowed:
    assert BEST_SOURCE_DIR is not None
    source_result_path = BEST_SOURCE_DIR / "trial_result.json"
    source_result = read_json(source_result_path)
    source_checkpoint = Path(source_result["checkpoint_best"])
    assert source_checkpoint.is_file()

    experiment_best_result_path = EXPERIMENT_BEST_DIR / "best_result.json"
    experiment_best_checkpoint = EXPERIMENT_BEST_DIR / "checkpoint_best.pth"

    previous_experiment_best = None
    if experiment_best_result_path.is_file():
        previous_experiment_best = float(
            read_json(experiment_best_result_path)["validation_micro_dice"]
        )

    EXPERIMENT_BEST_UPDATED = (
        UPDATE_EXPERIMENT_BEST
        and (
            previous_experiment_best is None
            or BEST_VALUE > previous_experiment_best
        )
    )

    if EXPERIMENT_BEST_UPDATED:
        shutil.copy2(source_checkpoint, experiment_best_checkpoint)
        experiment_payload = {
            "role": "EXPERIMENT_BEST",
            "experiment_id": EXPERIMENT_ID,
            "source_kind": "OPTUNA" if RUN_MODE == "OPTUNA" else "MANUAL",
            "source_run": (
                f"TRIAL_{BEST_TRIAL_NUMBER:03d}"
                if RUN_MODE == "OPTUNA"
                else MANUAL_RUN_ID
            ),
            "context_slices": NUM_CONTEXT_SLICES,
            "dataset_id": DATASET_ID,
            "dataset_name": DATASET_NAME,
            "base_model": "Base_Model",
            "base_source_legacy_experiment": "Exp05",
            "validation_micro_dice": BEST_VALUE,
            "checkpoint_best": str(experiment_best_checkpoint),
            "source_checkpoint": str(source_checkpoint),
            "source_dir": str(BEST_SOURCE_DIR),
            "postprocess_config": source_result["postprocess_config"],
            "updated_at": datetime.now().isoformat(timespec="seconds"),
        }
        write_json(experiment_best_result_path, experiment_payload)

    current_payload = read_json(CURRENT_BEST_JSON)
    previous_global_best = float(current_payload["validation_micro_dice"])

    GLOBAL_BEST_UPDATED = (
        UPDATE_GLOBAL_BEST
        and BEST_VALUE > previous_global_best
    )

    if GLOBAL_BEST_UPDATED:
        if BACKUP_PREVIOUS_BEST:
            backup_dir = BEST_HISTORY_DIR / (
                "BEST_BACKUP_"
                f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_"
                f"Dice_{previous_global_best:.6f}"
            )
            shutil.copytree(CURRENT_BEST_DIR, backup_dir)

        candidate_model_store = CURRENT_BEST_DIR / "nnUNet_model"
        if candidate_model_store.exists():
            shutil.rmtree(candidate_model_store)

        source_model_dir = Path(source_result["model_dir"])
        shutil.copytree(source_model_dir, candidate_model_store)

        current_checkpoint = CURRENT_BEST_DIR / "checkpoint_best.pth"
        shutil.copy2(source_checkpoint, current_checkpoint)

        new_current_payload = {
            "role": "CURRENT_BEST",
            "model_name": EXPERIMENT_ID,
            "source_experiment": EXPERIMENT_ID,
            "source_run": (
                f"TRIAL_{BEST_TRIAL_NUMBER:03d}"
                if RUN_MODE == "OPTUNA"
                else MANUAL_RUN_ID
            ),
            "context_slices": NUM_CONTEXT_SLICES,
            "dataset_id": DATASET_ID,
            "dataset_name": DATASET_NAME,
            "validation_micro_dice": BEST_VALUE,
            "checkpoint_best": str(current_checkpoint),
            "nnunet_model_dir": str(candidate_model_store),
            "postprocess_config": source_result["postprocess_config"],
            "base_reference_validation_micro_dice": 0.690981,
            "base_reference_test_micro_dice": 0.712285,
            "updated_at": datetime.now().isoformat(timespec="seconds"),
        }
        write_json(CURRENT_BEST_JSON, new_current_payload)

    print("기존 Experiment Best:", previous_experiment_best)
    print("현재 Validation Dice :", BEST_VALUE)
    print("Experiment Best 갱신 :", EXPERIMENT_BEST_UPDATED)
    print("기존 Current Best    :", previous_global_best)
    print("Current Best 갱신    :", GLOBAL_BEST_UPDATED)
else:
    print("현재 RUN_MODE에서는 Best Registry를 갱신하지 않습니다.")

print("✅ CELL 11 완료")


In [ ]:
# ==================================================================================================
# CELL 12. 실행 요약·로그 저장 및 Base 불변 검증
# ==================================================================================================

BASE_CHECKPOINT_HASH_AFTER = sha256_file(BASE_CHECKPOINT)
BASE_TRAINER_HASH_AFTER = sha256_file(BASE_TRAINER_FILE)

assert BASE_CHECKPOINT_HASH_BEFORE == BASE_CHECKPOINT_HASH_AFTER, (
    "Base checkpoint가 변경되었습니다. 실행을 중단하세요."
)
assert BASE_TRAINER_HASH_BEFORE == BASE_TRAINER_HASH_AFTER, (
    "Base Trainer가 변경되었습니다. 실행을 중단하세요."
)

if RUN_MODE in {"SMOKE", "MANUAL", "RESUME", "OPTUNA"}:
    assert BEST_SOURCE_DIR is not None
    source_result = read_json(BEST_SOURCE_DIR / "trial_result.json")

    summary = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "experiment_id": EXPERIMENT_ID,
        "run_mode": RUN_MODE,
        "manual_run_id": MANUAL_RUN_ID,
        "profile": PROFILE,
        "optuna_stage": OPTUNA_STAGE if RUN_MODE == "OPTUNA" else None,
        "pruner": PRUNER_NAME if RUN_MODE == "OPTUNA" else None,
        "context_slices": NUM_CONTEXT_SLICES,
        "dataset_id": DATASET_ID,
        "dataset_name": DATASET_NAME,
        "base_model": "Base_Model",
        "base_source_legacy_experiment": "Exp05",
        "use_base_pretrained_weights": USE_BASE_PRETRAINED_WEIGHTS,
        "loss": source_result["config"]["loss"],
        "optimizer": source_result["config"]["optimizer"],
        "learning_rate": source_result["config"]["learning_rate"],
        "weight_decay": source_result["config"]["weight_decay"],
        "scheduler": source_result["config"]["scheduler"],
        "batch_size": source_result["config"]["batch_size"],
        "oversample_foreground_percent": source_result["config"][
            "oversample_foreground_percent"
        ],
        "positive_case_ratio": source_result["config"]["positive_case_ratio"],
        "epochs": source_result["config"]["trial_epochs"],
        "validation_micro_dice": BEST_VALUE,
        "experiment_best_updated": EXPERIMENT_BEST_UPDATED,
        "global_best_updated": GLOBAL_BEST_UPDATED,
        "source_dir": str(BEST_SOURCE_DIR),
        "source_checkpoint": source_result["checkpoint_best"],
    }

    summary_dir = BEST_SOURCE_DIR / "summary"
    summary_dir.mkdir(parents=True, exist_ok=True)
    write_json(summary_dir / "final_summary.json", summary)

    history_csv = PATHS["logs"] / "experiment_history.csv"
    new_df = pd.DataFrame([summary])
    if history_csv.is_file():
        old_df = pd.read_csv(history_csv)
        history_df = pd.concat([old_df, new_df], ignore_index=True)
    else:
        history_df = new_df
    history_df.to_csv(
        history_csv,
        index=False,
        encoding="utf-8-sig",
    )

    print("=" * 110)
    print("팀 공용 nnU-Net 실행 완료")
    print("=" * 110)
    print("Experiment :", EXPERIMENT_ID)
    print("Run Mode   :", RUN_MODE)
    print("Source     :", BEST_SOURCE_DIR)
    print("Validation :", BEST_VALUE)
    print("Exp Best   :", EXPERIMENT_BEST_UPDATED)
    print("Global Best:", GLOBAL_BEST_UPDATED)
    print("History CSV:", history_csv)
    print("=" * 110)

elif RUN_MODE == "BASE_EVALUATE":
    print("BASE_EVALUATE 완료: Drive 저장 없음, Base 변경 없음.")

elif RUN_MODE in {"POSTPROCESS", "TEST_FINAL"}:
    print(f"{RUN_MODE} 완료. Base_Model 변경 없음.")

print("Base checkpoint SHA256:", BASE_CHECKPOINT_HASH_AFTER)
print("Base Trainer SHA256   :", BASE_TRAINER_HASH_AFTER)
print("✅ Base_Model 불변 확인 완료")


# 사용 순서

1. `BASE_EVALUATE`
   - Drive에는 평가 결과를 저장하지 않는다.
   - Validation `0.690981`, Test `0.712285` 재현 여부를 확인한다.

2. `SMOKE`
   - `EXP_NO=1`, 2 epoch
   - `Exp01/manual_runs/RUN_000`에 결과가 저장된다.
   - 기본 설정에서는 Current Best를 갱신하지 않는다.

3. 실제 실험
   - Fine-tuning: `RUN_MODE="MANUAL"`, `USE_BASE_PRETRAINED_WEIGHTS=True`
   - Scratch: `RUN_MODE="MANUAL"`, `USE_BASE_PRETRAINED_WEIGHTS=False`
   - Resume: `RUN_MODE="RESUME"`, `RESUME_RUN_ID="RUN_000"`
   - Optuna: `RUN_MODE="OPTUNA"` 및 Search Space/Pruner 설정

4. `POSTPROCESS`
   - 저장된 Validation probability를 이용해 Threshold/Component Size를 다시 탐색한다.

5. `TEST_FINAL`
   - 최종 후보 1개에만 Test를 수행한다.
   - Test 성능은 Optuna 또는 Best 선택 기준으로 사용하지 않는다.
